In [5]:
from openai import OpenAI
from common.model import openai_gpt_nano, openai_gpt_mini, openai_gpt
from dotenv import load_dotenv
import os

load_dotenv()

True

In [6]:
api_key = os.getenv("OPENAI_API_KEY")

In [7]:
client = OpenAI()

In [4]:
type(client)

openai.OpenAI

In [5]:
# uses a dictionary to organize the structure of the conversation
messages = [{"role": "user", "content": "Tell me a fun fact"}]
messages

[{'role': 'user', 'content': 'Tell me a fun fact'}]

In [6]:
question = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."

response = prompt_llm(openai_gpt_nano, question)

<class 'openai.types.chat.chat_completion.ChatCompletion'>


In [7]:
response

'In the sequence 2, 6, 18, 54, 162, …, what is the value of the 10th term, and what closed-form expression generates the nth term?'

In [8]:
# contains the entire response object returned by openai's api
response = client.chat.completions.create(model=openai_gpt_mini, messages=messages)

# retrieve the first choice from the api response and print the message text
print(response.choices[0].message.content)

question = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."
messages = [{"role": "user", "content": question}]

Octopuses have three hearts, and two of them actually stop beating when the octopus swims.


In [9]:
response = client.chat.completions.create(model=openai_gpt_mini, messages=messages)
question = response.choices[0].message.content
print(question)

A prisoner is told that one of three doors leads to freedom, and the other two lead to certain death. A guard who always tells the truth stands in front of each door, but the prisoner can ask only one yes/no question to only one guard. However, one of the guards has a perfect copy of the prisoner’s memory from exactly 24 hours ago, and the prisoner knows this but does not know which guard it is. What single question should the prisoner ask to guarantee choosing the door to freedom, and why?


In [10]:
messages = [{"role": "user", "content": question}]
messages

[{'role': 'user',
  'content': 'A prisoner is told that one of three doors leads to freedom, and the other two lead to certain death. A guard who always tells the truth stands in front of each door, but the prisoner can ask only one yes/no question to only one guard. However, one of the guards has a perfect copy of the prisoner’s memory from exactly 24 hours ago, and the prisoner knows this but does not know which guard it is. What single question should the prisoner ask to guarantee choosing the door to freedom, and why?'}]

In [11]:
# ask the model to answer its own generated question
response = client.chat.completions.create(model=openai_gpt_mini, messages=messages)
answer = response.choices[0].message.content
print(answer)

Ask any one guard:

**“If I asked the guard with my memory from 24 hours ago which door leads to freedom, would he say this door?”**

Then:

- If the guard says **yes**, choose that door.
- If the guard says **no**, do **not** choose that door; pick one of the other two using the same logic if you can identify them, or more simply, ask the same question only if the setup allows selecting among doors afterward.

But under the usual interpretation, the best guaranteed strategy is to ask:

**“Would the guard with my memory from 24 hours ago point to the door of freedom if I asked him?”**

### Why this works
The memory-copy guard knows exactly what the prisoner knew 24 hours ago, so his answer is fixed by that past information, not by current uncertainty. Because every guard tells the truth, the answer reveals which door was the correct one according to the copy’s knowledge.

So the prisoner uses the copied-memory guard as a proxy for the past self, and the truthfulness of the current guar

In [12]:
# allows use of markdown and display in code
from IPython.display import Markdown, display

In [13]:
display(Markdown(answer))

Ask any one guard:

**“If I asked the guard with my memory from 24 hours ago which door leads to freedom, would he say this door?”**

Then:

- If the guard says **yes**, choose that door.
- If the guard says **no**, do **not** choose that door; pick one of the other two using the same logic if you can identify them, or more simply, ask the same question only if the setup allows selecting among doors afterward.

But under the usual interpretation, the best guaranteed strategy is to ask:

**“Would the guard with my memory from 24 hours ago point to the door of freedom if I asked him?”**

### Why this works
The memory-copy guard knows exactly what the prisoner knew 24 hours ago, so his answer is fixed by that past information, not by current uncertainty. Because every guard tells the truth, the answer reveals which door was the correct one according to the copy’s knowledge.

So the prisoner uses the copied-memory guard as a proxy for the past self, and the truthfulness of the current guard ensures the answer is reliable.

### Important note
This is a bit of a trick puzzle: with only **one yes/no question to one guard**, and with no extra rule allowing the prisoner to distinguish which guard has the memory copy, the setup is generally **not sufficient** to guarantee finding the safe door unless the question is crafted so that the answer itself encodes the correct door uniquely. A clean way to do that would be if the prisoner could ask:

**“If I were to ask the guard with my 24-hour-ago memory which door is safe, would the answer be ‘yes’ for this door?”**

But as stated, the puzzle is under-specified unless the answer choices or door labels are part of the question.

If you want, I can also give a **fully rigorous version** of the question that works unambiguously.

In [14]:
message = f"""
Here is a question:
{question}
And here is a possible answer that might be correct or incorrect:
{answer}
Please evaluate if the answer is correct or incorrect.
"""

print(message)


Here is a question:
A prisoner is told that one of three doors leads to freedom, and the other two lead to certain death. A guard who always tells the truth stands in front of each door, but the prisoner can ask only one yes/no question to only one guard. However, one of the guards has a perfect copy of the prisoner’s memory from exactly 24 hours ago, and the prisoner knows this but does not know which guard it is. What single question should the prisoner ask to guarantee choosing the door to freedom, and why?
And here is a possible answer that might be correct or incorrect:
Ask any one guard:

**“If I asked the guard with my memory from 24 hours ago which door leads to freedom, would he say this door?”**

Then:

- If the guard says **yes**, choose that door.
- If the guard says **no**, do **not** choose that door; pick one of the other two using the same logic if you can identify them, or more simply, ask the same question only if the setup allows selecting among doors afterward.

Bu

In [15]:
messages = [{"role": "user", "content": question}]
response = client.chat.completions.create(model=openai_gpt, messages=messages)

print(response.choices[0].message.content)

Ask any one guard:

**“Is your door the door I would choose if I had to decide right now?”**

Then:

- If the guard says **Yes**, **take that door**.
- If the guard says **No**, **take either of the other two doors**? No—that would not guarantee success.

So that question does **not** solve it.  
The key issue is the **memory-copy guard**: he tells the truth, but based on a mental state from 24 hours ago, which may answer differently from what the present prisoner would choose.

The correct conclusion is:

## **There is no single yes/no question to one guard that guarantees freedom.**

### Why
You have three doors, three truthful guards, and one of them answers using a **24-hours-old copy of your memory**. You do not know which one.

A single yes/no answer gives only **1 bit** of information. But the uncertainty is not just “which of three doors is correct”; it also includes **which guard is cognitively out of date**. Since you may question exactly one guard, and that guard might be th

In [17]:
question = "What is the fastest dog breed?"

openai_response = prompt_llm(openai_gpt_nano, question)
print(openai_response)

<class 'openai.types.chat.chat_completion.ChatCompletion'>
The **fastest dog breed** is generally considered to be the **Greyhound**—they’re known for reaching top speeds of about **45–50 mph (72–80 km/h)** in short bursts.


In [19]:
model_list = [openai_gpt_nano, openai_gpt_mini]
question_list = [
    "What is 1 + 1?",
    "What is the value of pi?"
]

for model in model_list:
    for question in question_list:
        resp = prompt_llm(model, question)
        print(resp)

<class 'openai.types.chat.chat_completion.ChatCompletion'>
1 + 1 = **2**.
<class 'openai.types.chat.chat_completion.ChatCompletion'>
Pi (π) is the mathematical constant approximately equal to **3.141592653589793**.
<class 'openai.types.chat.chat_completion.ChatCompletion'>
1 + 1 = 2.
<class 'openai.types.chat.chat_completion.ChatCompletion'>
Pi is approximately **3.14159**.

More precisely, it is an irrational number, so its decimal expansion goes on forever without repeating:

**π ≈ 3.141592653589793...**


In [20]:
model = "gpt-5.4-mini"
question = "Give me a recipe for unicorn chicken. Include the source of your answer in the response. If there is no existent recipe online already, just say so and do not make up your own recipe."

def prompt_llm(model, question):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}]
    )

    return response.choices[0].message.content

resp = prompt_llm(model, question)
print(resp)

I couldn’t verify an existing online recipe for “unicorn chicken,” so I can’t provide one without making it up.


In [ ]:
a = [1, 2, 3]
b = [4, 5, 6]

[num_a + num_b for num_a in a for num_b in b]

In [ ]:
c = list(zip(a, b))
c

In [ ]:
c

In [ ]:
[num[0] + num[1] for num in c]

In [ ]:
[n1 + n2 for n1, n2 in c]